In [4]:
# ============================================================
# SFT TRAINING (E0) — ALL 6 QWEN2.5 MODELS
# - Pure SFT (no WSFT, no confidence)
# - Teacher forcing on teacher_text
# - Prompt tokens masked (labels = -100)
# - ✅ Uses ONLY the frozen train set: clinical_pharm_teacher_train_6000.jsonl
# ============================================================

!pip -q install -U transformers peft accelerate bitsandbytes sentencepiece

import os, json, random
from dataclasses import dataclass
from typing import Dict, List

import torch
from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)

from peft import LoraConfig, get_peft_model
from google.colab import drive

drive.mount("/content/drive")
# ----------------------------
# CONFIG
# ----------------------------

BASE_DIR = "/content/drive/MyDrive/DL_Local"
PROMPTS_PATH = os.path.join(BASE_DIR, "clinical_pharm_prompts_10000.jsonl")

# ✅ CHANGED: use frozen subset instead of full teacher JSON
TEACHER_6K_PATH = os.path.join(BASE_DIR, "clinical_pharm_teacher_train_6000.jsonl")

OUT_DIR = os.path.join(BASE_DIR, "sft_runs")

MAX_SEQ_LEN = 2048
EPOCHS = 2
LR = 2e-4
BATCH_SIZE = 1
GRAD_ACC = 32
SEED = 42

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

STUDENTS = {
    "qwen25_1p5b_base":     {"path": "Qwen/Qwen2.5-1.5B",          "is_instruct": False},
    "qwen25_1p5b_instruct": {"path": "Qwen/Qwen2.5-1.5B-Instruct", "is_instruct": True},
    "qwen25_3b_base":       {"path": "Qwen/Qwen2.5-3B",            "is_instruct": False},
    "qwen25_3b_instruct":   {"path": "Qwen/Qwen2.5-3B-Instruct",   "is_instruct": True},
    "qwen25_7b_base":       {"path": "Qwen/Qwen2.5-7B",            "is_instruct": False},
    "qwen25_7b_instruct":   {"path": "Qwen/Qwen2.5-7B-Instruct",   "is_instruct": True},
}

# ----------------------------
# DATA LOADING
# ----------------------------

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

prompts = {r["id"]: r["prompt"] for r in load_jsonl(PROMPTS_PATH)}

# ✅ CHANGED: load teacher rows ONLY from frozen subset
teacher_rows = [
    r for r in load_jsonl(TEACHER_6K_PATH)
    if (
        r.get("status") == "ok"
        and r.get("teacher_text")
        # split is already train, but keep this guard
        and r.get("split") == "train"
    )
]

print(f"Loaded {len(teacher_rows)} teacher samples (FROZEN TRAIN SET) from: {TEACHER_6K_PATH}")

# ----------------------------
# DATASET
# ----------------------------

class SFTDataset(Dataset):
    def __init__(self, rows, tokenizer, is_instruct):
        self.rows = rows
        self.tokenizer = tokenizer
        self.is_instruct = is_instruct

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        prompt = prompts[r["id"]]
        answer = r["teacher_text"]

        if self.is_instruct and hasattr(self.tokenizer, "apply_chat_template"):
            prompt_text = self.tokenizer.apply_chat_template(
                [{"role": "user", "content": prompt}],
                tokenize=False,
                add_generation_prompt=True
            )
        else:
            prompt_text = prompt

        full_text = prompt_text + answer

        tok = self.tokenizer(
            full_text,
            truncation=True,
            max_length=MAX_SEQ_LEN,
            return_offsets_mapping=True
        )

        input_ids = tok["input_ids"]
        offsets = tok["offset_mapping"]

        labels = [-100] * len(input_ids)

        # find where answer starts
        answer_start = len(prompt_text)
        for i, (s, e) in enumerate(offsets):
            if s >= answer_start:
                labels[i] = input_ids[i]

        return {
            "input_ids": torch.tensor(input_ids),
            "labels": torch.tensor(labels),
            "attention_mask": torch.ones(len(input_ids))
        }

# ----------------------------
# TRAINING LOOP
# ----------------------------

for model_name, cfg in STUDENTS.items():
    print(f"\n===== TRAINING {model_name} =====")

    out_path = os.path.join(OUT_DIR, model_name)
    os.makedirs(out_path, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(cfg["path"], trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        cfg["path"],
        load_in_4bit=True,
        device_map="auto",
        trust_remote_code=True
    )

    lora_cfg = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )

    model = get_peft_model(model, lora_cfg)
    model.train()

    dataset = SFTDataset(teacher_rows, tokenizer, cfg["is_instruct"])

    args = TrainingArguments(
        output_dir=out_path,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACC,
        learning_rate=LR,
        logging_steps=50,
        save_strategy="no",
        fp16=True,
        seed=SEED,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=dataset,
        data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    )

    trainer.train()
    model.save_pretrained(out_path)
    tokenizer.save_pretrained(out_path)

    print(f"✅ Finished {model_name}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded 5000 teacher samples (FROZEN TRAIN SET) from: /content/drive/MyDrive/DL_Local/clinical_pharm_teacher_train_6000.jsonl

===== TRAINING qwen25_1p5b_base =====


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Step,Training Loss
50,0.995500
100,0.644100
150,0.594600
200,0.564600
250,0.549900
300,0.542700


✅ Finished qwen25_1p5b_base

===== TRAINING qwen25_1p5b_instruct =====


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


...

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Step,Training Loss
50,0.866800
100,0.492500
150,0.454900
200,0.430200
250,0.418500
300,0.411300


✅ Finished qwen25_7b_instruct


In [7]:
# ============================================================
# WSFT TRAINING (E1) — ALL 6 QWEN2.5 MODELS
# - Reasoning-weighted SFT (Explanation + Decision upweighted)
# - Teacher forcing on teacher_text
# - Prompt tokens masked (labels = -100)
# - ✅ Uses ONLY frozen train set: clinical_pharm_teacher_train_6000.jsonl
# - ✅ Token-level weighted loss (format=1, decision=2, explanation=2)
# - ✅ Optional per-sample weight normalization (mean weight over answer tokens = 1)
# ============================================================

!pip -q install -U transformers peft accelerate bitsandbytes sentencepiece

import os, json, re
from typing import List, Tuple, Dict, Any

import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model
from google.colab import drive

drive.mount("/content/drive")
# ----------------------------
# CONFIG
# ----------------------------
BASE_DIR = "/content/drive/MyDrive/DL_Local"
PROMPTS_PATH = os.path.join(BASE_DIR, "clinical_pharm_prompts_10000.jsonl")
TEACHER_6K_PATH = os.path.join(BASE_DIR, "clinical_pharm_teacher_train_6000.jsonl")

OUT_DIR = os.path.join(BASE_DIR, "wsft_runs")  # ✅ separate folder from E0
os.makedirs(OUT_DIR, exist_ok=True)

MAX_SEQ_LEN = 2048
EPOCHS = 2
LR = 2e-4
BATCH_SIZE = 1
GRAD_ACC = 32
SEED = 42

# WSFT weights
W_FORMAT = 1.0
W_DECISION = 2.0
W_EXPL = 2.0

# If True: normalize per-sample weights so mean(weight over supervised tokens) == 1
NORMALIZE_WEIGHTS_PER_SAMPLE = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

STUDENTS = {
    "qwen25_1p5b_base":     {"path": "Qwen/Qwen2.5-1.5B",          "is_instruct": False},
    "qwen25_1p5b_instruct": {"path": "Qwen/Qwen2.5-1.5B-Instruct", "is_instruct": True},
    "qwen25_3b_base":       {"path": "Qwen/Qwen2.5-3B",            "is_instruct": False},
    "qwen25_3b_instruct":   {"path": "Qwen/Qwen2.5-3B-Instruct",   "is_instruct": True},
    "qwen25_7b_base":       {"path": "Qwen/Qwen2.5-7B",            "is_instruct": False},
    "qwen25_7b_instruct":   {"path": "Qwen/Qwen2.5-7B-Instruct",   "is_instruct": True},
}

# ----------------------------
# DATA LOADING
# ----------------------------
def load_jsonl(path: str) -> List[Dict[str, Any]]:
    out = []
    with open(path, "r", encoding="utf-8") as f:
        for l in f:
            l = l.strip()
            if not l:
                continue
            out.append(json.loads(l))
    return out

prompts = {r["id"]: r["prompt"] for r in load_jsonl(PROMPTS_PATH)}

teacher_rows = [
    r for r in load_jsonl(TEACHER_6K_PATH)
    if (r.get("status") == "ok" and r.get("teacher_text") and r.get("split") == "train")
]

print(f"Loaded {len(teacher_rows)} teacher samples (FROZEN TRAIN SET) from: {TEACHER_6K_PATH}")

# ----------------------------
# SECTION SPAN FINDER (robust heuristic)
# Supports:
# - JSON-like keys: "decision": ..., "explanation": ...
# - Headings: Decision: ... / Explanation: ...
# Returns spans as (start_char, end_char) within answer text
# ----------------------------
_DECISION_PATTERNS = [
    r'"decision"\s*:\s*', r"'decision'\s*:\s*", r"\bDecision\s*:\s*", r"\bDECISION\s*:\s*"
]
_EXPL_PATTERNS = [
    r'"explanation"\s*:\s*', r"'explanation'\s*:\s*", r"\bExplanation\s*:\s*", r"\bEXPLANATION\s*:\s*",
    r'"reasoning"\s*:\s*', r"'reasoning'\s*:\s*", r"\bReasoning\s*:\s*", r"\bREASONING\s*:\s*"
]

def _find_span_by_markers(text: str, start_markers: List[str], end_markers: List[str]) -> Tuple[int, int]:
    """
    Find span starting after the first matched start marker, ending at the next matched end marker.
    If no end marker found, span goes to end of text.
    Returns (-1,-1) if not found.
    """
    start_idx = -1
    start_end = -1

    for pat in start_markers:
        m = re.search(pat, text)
        if m:
            start_idx = m.start()
            start_end = m.end()
            break
    if start_end == -1:
        return (-1, -1)

    # Find the earliest next end marker AFTER start_end
    end_pos = len(text)
    for pat in end_markers:
        m2 = re.search(pat, text[start_end:])
        if m2:
            end_pos = min(end_pos, start_end + m2.start())
    return (start_end, end_pos)

def get_section_spans(answer_text: str) -> Dict[str, List[Tuple[int, int]]]:
    """
    Returns dict with keys: 'decision', 'explanation'.
    Each maps to a list of spans (start,end) in answer_text.
    """
    spans = {"decision": [], "explanation": []}

    # Decision span ends when explanation starts (or end)
    d_span = _find_span_by_markers(answer_text, _DECISION_PATTERNS, _EXPL_PATTERNS)
    if d_span != (-1, -1) and d_span[0] < d_span[1]:
        spans["decision"].append(d_span)

    # Explanation span ends when decision starts (if decision appears after) OR end
    e_span = _find_span_by_markers(answer_text, _EXPL_PATTERNS, _DECISION_PATTERNS)
    if e_span != (-1, -1) and e_span[0] < e_span[1]:
        spans["explanation"].append(e_span)

    # If both exist and overlap weirdly, we keep them anyway (weights will just apply where tokens fall).
    return spans

# ----------------------------
# DATASET (returns per-token weights)
# ----------------------------
class WSFTDataset(Dataset):
    def __init__(self, rows, tokenizer, is_instruct):
        self.rows = rows
        self.tokenizer = tokenizer
        self.is_instruct = is_instruct

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        ex_id = r["id"]
        prompt = prompts[ex_id]
        answer = r["teacher_text"]

        # build prompt_text (match E0 behavior)
        if self.is_instruct and hasattr(self.tokenizer, "apply_chat_template"):
            prompt_text = self.tokenizer.apply_chat_template(
                [{"role": "user", "content": prompt}],
                tokenize=False,
                add_generation_prompt=True
            )
        else:
            prompt_text = prompt

        full_text = prompt_text + answer

        tok = self.tokenizer(
            full_text,
            truncation=True,
            max_length=MAX_SEQ_LEN,
            return_offsets_mapping=True
        )
        input_ids = tok["input_ids"]
        offsets = tok["offset_mapping"]

        # labels: mask prompt tokens, supervise only answer tokens
        labels = [-100] * len(input_ids)
        weights = [0.0] * len(input_ids)  # 0 for masked tokens

        answer_start_abs = len(prompt_text)  # char index in full_text where answer begins

        # Compute answer section spans in answer-local coords, then convert to full_text coords
        spans = get_section_spans(answer)
        decision_spans_abs = [(answer_start_abs + s, answer_start_abs + e) for (s, e) in spans["decision"]]
        expl_spans_abs     = [(answer_start_abs + s, answer_start_abs + e) for (s, e) in spans["explanation"]]

        def in_any_span(token_s: int, token_e: int, span_list: List[Tuple[int,int]]) -> bool:
            # overlap check (token span intersects section span)
            for (s, e) in span_list:
                if token_e > s and token_s < e:
                    return True
            return False

        for i, (s, e) in enumerate(offsets):
            # ignore special tokens that may have (0,0)
            if e <= s:
                continue

            # supervised tokens = those starting at/after answer start
            if s >= answer_start_abs:
                labels[i] = input_ids[i]

                # determine WSFT weight
                w = W_FORMAT
                if in_any_span(s, e, decision_spans_abs):
                    w = W_DECISION
                if in_any_span(s, e, expl_spans_abs):
                    # if both match, explanation wins (same weight anyway in our chosen config)
                    w = W_EXPL

                weights[i] = float(w)

        attention_mask = [1] * len(input_ids)

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "weights": torch.tensor(weights, dtype=torch.float),
        }

# ----------------------------
# COLLATOR (pads weights + labels properly)
# ----------------------------
class WSFTCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features: List[Dict[str, torch.Tensor]]) -> Dict[str, torch.Tensor]:
        # Convert to python lists for tokenizer.pad
        batch = {
            "input_ids": [f["input_ids"] for f in features],
            "attention_mask": [f["attention_mask"] for f in features],
            "labels": [f["labels"] for f in features],
            "weights": [f["weights"] for f in features],
        }

        # Pad input_ids/attention_mask with tokenizer.pad
        padded = self.tokenizer.pad(
            {"input_ids": batch["input_ids"], "attention_mask": batch["attention_mask"]},
            padding=True,
            return_tensors="pt"
        )

        # Manually pad labels + weights to same length
        max_len = padded["input_ids"].shape[1]

        def pad_1d(seq_list, pad_value, dtype):
            out = torch.full((len(seq_list), max_len), pad_value, dtype=dtype)
            for i, s in enumerate(seq_list):
                L = s.shape[0]
                out[i, :L] = s.to(dtype)
            return out

        padded["labels"] = pad_1d(batch["labels"], pad_value=-100, dtype=torch.long)
        padded["weights"] = pad_1d(batch["weights"], pad_value=0.0, dtype=torch.float)

        return padded

# ----------------------------
# CUSTOM TRAINER (weighted token loss)
# ----------------------------
class WeightedLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        weights = inputs.pop("weights")
        outputs = model(**inputs)
        logits = outputs.logits  # [B, T, V]

        # Shift for causal LM
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()
        shift_weights = weights[:, 1:].contiguous()

        # Compute token loss (no reduction)
        vocab_size = shift_logits.size(-1)
        loss_fct = torch.nn.CrossEntropyLoss(reduction="none")

        # flatten
        loss_flat = loss_fct(
            shift_logits.view(-1, vocab_size),
            shift_labels.view(-1)
        ).view(shift_labels.size())  # [B, T-1]

        # mask: only supervised tokens (labels != -100)
        active = (shift_labels != -100).float()

        # apply weights only on active tokens
        w = shift_weights * active

        if NORMALIZE_WEIGHTS_PER_SAMPLE:
            # normalize so mean weight over active tokens == 1 (per sample)
            # avoid div0
            denom = active.sum(dim=1, keepdim=True).clamp(min=1.0)
            mean_w = (w.sum(dim=1, keepdim=True) / denom).clamp(min=1e-6)
            w = w / mean_w

        weighted_loss = loss_flat * w

        # final reduction: sum over active tokens / count active tokens
        denom_all = active.sum().clamp(min=1.0)
        loss = weighted_loss.sum() / denom_all

        return (loss, outputs) if return_outputs else loss

# ----------------------------
# TRAINING LOOP
# ----------------------------
for model_name, cfg in STUDENTS.items():
    print(f"\n===== WSFT TRAINING {model_name} =====")

    out_path = os.path.join(OUT_DIR, model_name)
    os.makedirs(out_path, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(cfg["path"], trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        cfg["path"],
        load_in_4bit=True,
        device_map="auto",
        trust_remote_code=True
    )

    lora_cfg = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )

    model = get_peft_model(model, lora_cfg)
    model.train()

    dataset = WSFTDataset(teacher_rows, tokenizer, cfg["is_instruct"])
    collator = WSFTCollator(tokenizer)

    args = TrainingArguments(
        output_dir=out_path,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACC,
        learning_rate=LR,
        logging_steps=50,
        save_strategy="no",
        fp16=True,
        seed=SEED,
        report_to="none",
        remove_unused_columns=False,

    )

    trainer = WeightedLossTrainer(
        model=model,
        args=args,
        train_dataset=dataset,
        data_collator=collator,
    )

    trainer.train()
    model.save_pretrained(out_path)
    tokenizer.save_pretrained(out_path)

    print(f"✅ Finished WSFT {model_name}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded 5000 teacher samples (FROZEN TRAIN SET) from: /content/drive/MyDrive/DL_Local/clinical_pharm_teacher_train_6000.jsonl

===== WSFT TRAINING qwen25_1p5b_base =====


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss
50,43.317700
100,36.717100
150,34.284700
200,32.067300
250,31.737400
300,31.440900


✅ Finished WSFT qwen25_1p5b_base

===== WSFT TRAINING qwen25_1p5b_instruct =====


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


...

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss
50,35.192000
100,29.204200
150,27.207900
200,25.073700
250,24.705700
300,24.431100


✅ Finished WSFT qwen25_7b_instruct
